
#### GOLD LAYER - Business Insights & Analytics

  Create Dim_Customers Table (SCD Type 2)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import monotonically_increasing_id, col, current_date, lit, concat, datediff, when

# Read Silver tables
customers_silver = spark.table("banking_catalog.silver.customers")
accounts_silver = spark.table("banking_catalog.silver.accounts")
transactions_silver = spark.table("banking_catalog.silver.transactions")
loans_silver = spark.table("banking_catalog.silver.loans")
fraud_silver = spark.table("banking_catalog.silver.fraud_alerts")
atm_silver = spark.table("banking_catalog.silver.atm_transactions")
branches_silver = spark.table("banking_catalog.silver.branches")

# Create Dim_Customers with SCD Type 2
dim_customers = customers_silver \
    .withColumn("customer_sk", monotonically_increasing_id()) \
    .withColumn("effective_date", current_date()) \
    .withColumn("end_date", lit("9999-12-31").cast("date")) \
    .withColumn("is_current", lit(True)) \
    .withColumn("full_name", concat(col("name"), lit(" (ID: "), col("customer_id"), lit(")"))) \
    .withColumn("age", datediff(current_date(), col("dob")) / 365.25) \
    .withColumn("age_group", 
        when(col("age") < 30, "Young")
        .when(col("age") < 50, "Middle")
        .otherwise("Senior"))

# Write to Gold
dim_customers.write.format("delta") \
    .mode("overwrite") \
    .save("abfss://gold@stbankinglake005.dfs.core.windows.net/dim_customers")

print(f"✅ Dim_Customers: {dim_customers.count()} records")

✅ Dim_Customers: 500 records



Create Dim_Accounts Table


In [0]:
%python
# Create Dim_Accounts
dim_accounts = accounts_silver \
    .withColumn("account_sk", monotonically_increasing_id()) \
    .withColumn("effective_date", current_date()) \
    .withColumn("is_current", lit(True)) \
    .withColumn("account_age_days", datediff(current_date(), col("date"))) \
    .withColumn("balance_category",
        when(col("amount") < 10000, "Low")
        .when(col("amount") < 50000, "Medium")
        .otherwise("High"))

# Write to Gold
dim_accounts.write.format("delta") \
    .mode("overwrite") \
    .save("abfss://gold@stbankinglake005.dfs.core.windows.net/dim_accounts")

print(f"✅ Dim_Accounts: {dim_accounts.count()} records")

✅ Dim_Accounts: 500 records



Create Dim_Date Table

In [0]:
%python
from pyspark.sql.functions import year, quarter, month, dayofmonth, dayofweek, date_format, when, col, monotonically_increasing_id

# Create Date Dimension (2020-2025)
def create_date_dimension(start_date, end_date):
    dates = spark.sql(f"""
        SELECT EXPLODE(SEQUENCE(
            TO_DATE('{start_date}'), 
            TO_DATE('{end_date}'), 
            INTERVAL 1 DAY
        )) AS date
    """)
    
    date_dim = dates.select(
        col("date"),
        year("date").alias("year"),
        quarter("date").alias("quarter"),
        month("date").alias("month"),
        dayofmonth("date").alias("day"),
        dayofweek("date").alias("day_of_week"),
        date_format("date", "EEEE").alias("day_name"),
        date_format("date", "MMMM").alias("month_name"),
        when(dayofweek("date").isin([1, 7]), "Weekend").otherwise("Weekday").alias("weekday_indicator")
    ).withColumn("date_sk", monotonically_increasing_id())
    
    return date_dim

date_dim_df = create_date_dimension("2020-01-01", "2025-12-31")

date_dim_df.write.format("delta") \
    .mode("overwrite") \
    .save("abfss://gold@stbankinglake005.dfs.core.windows.net/dim_date")

print(f"✅ Dim_Date: {date_dim_df.count()} records")


✅ Dim_Date: 2192 records


Create Fact_Transactions

In [0]:
# Create Fact_Transactions with surrogate keys
fact_transactions = transactions_silver \
    .join(dim_accounts.select("account_id", "account_sk"), "account_id", "left") \
    .join(dim_customers.select("customer_id", "customer_sk"), "customer_id", "left") \
    .join(date_dim_df.select("date", "date_sk"), transactions_silver["date"] == date_dim_df["date"], "left") \
    .select(
        "transaction_id",
        "customer_sk",
        "account_sk",
        "date_sk",
        "amount",
        "status",
        col("type").alias("transaction_type"),
        "branch",
        "remarks",
        "flag",
        "year",
        "month",
        "quarter"
    )

# Write to Gold
fact_transactions.write.format("delta") \
    .mode("overwrite") \
    .save("abfss://gold@stbankinglake005.dfs.core.windows.net/fact_transactions")

print(f"✅ Fact_Transactions: {fact_transactions.count()} records")


✅ Fact_Transactions: 500 records



Create Fact_Loans

In [0]:
%python
# Create Fact_Loans
fact_loans = loans_silver \
    .join(dim_accounts.select("account_id", "account_sk"), "account_id", "left") \
    .join(dim_customers.select("customer_id", "customer_sk"), "customer_id", "left") \
    .join(date_dim_df.select("date", "date_sk"), loans_silver["date"] == date_dim_df["date"], "left") \
    .select(
        "loan_id",
        "customer_sk",
        "account_sk",
        "date_sk",
        "amount",
        "status",
        col("type").alias("loan_type"),
        "branch",
        "remarks",
        "flag"
    )

# Write to Gold
fact_loans.write.format("delta") \
    .mode("overwrite") \
    .save("abfss://gold@stbankinglake005.dfs.core.windows.net/fact_loans")

print(f"✅ Fact_Loans: {fact_loans.count()} records")


✅ Fact_Loans: 500 records


Create Fact_Fraud_Alerts

In [0]:
%python
# Create Fact_Fraud_Alerts
fact_fraud = fraud_silver \
    .join(dim_customers.select("customer_id", "customer_sk"), "customer_id", "left") \
    .join(dim_accounts.select("account_id", "account_sk"), "account_id", "left") \
    .join(date_dim_df.select("date", "date_sk"), fraud_silver["date"] == date_dim_df["date"], "left") \
    .select(
        "alert_id",
        "customer_sk",
        "account_sk",
        "date_sk",
        "amount",
        "status",
        col("type").alias("alert_type"),
        "branch",
        "remarks",
        "flag",
        "is_high_risk"
    )

# Write to Gold
fact_fraud.write.format("delta") \
    .mode("overwrite") \
    .save("abfss://gold@stbankinglake005.dfs.core.windows.net/fact_fraud_alerts")

print(f"✅ Fact_Fraud_Alerts: {fact_fraud.count()} records")


✅ Fact_Fraud_Alerts: 500 records


Create Gold Tables in Unity Catalog

In [0]:
%sql
-- Create Gold tables
CREATE TABLE banking_catalog.gold.dim_customers
USING DELTA
LOCATION 'abfss://gold@stbankinglake005.dfs.core.windows.net/dim_customers/';

CREATE TABLE banking_catalog.gold.dim_accounts
USING DELTA
LOCATION 'abfss://gold@stbankinglake005.dfs.core.windows.net/dim_accounts/';

CREATE TABLE banking_catalog.gold.dim_date
USING DELTA
LOCATION 'abfss://gold@stbankinglake005.dfs.core.windows.net/dim_date/';

CREATE TABLE banking_catalog.gold.fact_transactions
USING DELTA
LOCATION 'abfss://gold@stbankinglake005.dfs.core.windows.net/fact_transactions/';

CREATE TABLE banking_catalog.gold.fact_loans
USING DELTA
LOCATION 'abfss://gold@stbankinglake005.dfs.core.windows.net/fact_loans/';

CREATE TABLE banking_catalog.gold.fact_fraud_alerts
USING DELTA
LOCATION 'abfss://gold@stbankinglake005.dfs.core.windows.net/fact_fraud_alerts/';



## BUSINESS PROBLEMS SOLVED

Total Balance by Account Type

In [0]:
%sql
-- Problem: What is the total balance for each account type?
SELECT 
    type,
    COUNT(*) as account_count,
    SUM(amount) as total_balance,
    AVG(amount) as avg_balance,
    MIN(amount) as min_balance,
    MAX(amount) as max_balance
FROM banking_catalog.silver.accounts
WHERE amount IS NOT NULL
GROUP BY type
ORDER BY total_balance DESC;


type,account_count,total_balance,avg_balance,min_balance,max_balance
SAVINGS,182,4556796.900000002,25037.345604395618,0.0,49605.68
LOAN,169,4076798.0500000003,24123.065384615387,0.0,49681.93
CURRENT,149,3548266.6900000004,23813.870402684566,0.0,49977.11



Customer KYC Status Analysis

In [0]:
%sql
-- Problem: What percentage of customers have completed KYC?
SELECT 
    kyc_status,
    COUNT(*) as customer_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
FROM banking_catalog.silver.customers
GROUP BY kyc_status
ORDER BY percentage DESC;


kyc_status,customer_count,percentage
N,271,54.20
Y,229,45.80



Monthly Transaction Trends

In [0]:
%sql
-- Problem: How do transaction volumes trend monthly?
SELECT 
    YEAR(date) as year,
    MONTH(date) as month,
    COUNT(*) as transaction_count,
    SUM(amount) as total_amount,
    AVG(amount) as avg_amount
FROM banking_catalog.silver.transactions
GROUP BY YEAR(date), MONTH(date)
ORDER BY year DESC, month DESC;


year,month,transaction_count,total_amount,avg_amount
2025,6,4,134222.8,33555.7
2025,5,6,95725.71,15954.285000000002
2025,4,10,359357.83,35935.783
2025,3,5,186411.75,37282.35
2025,2,6,122327.87,20387.978333333333
2025,1,9,212572.42,23619.15777777778
2024,12,4,107669.87999999999,26917.469999999998
2024,11,4,113385.58,28346.395
2024,10,14,385690.17000000004,27549.29785714286
2024,9,9,209650.94,23294.54888888889



Top 10 Customers by Transaction Volume

In [0]:
%sql
-- Problem: Who are the top 10 customers by transaction volume?
SELECT 
    c.customer_id,
    c.name,
    COUNT(t.transaction_id) as transaction_count,
    SUM(t.amount) as total_transaction_amount,
    AVG(t.amount) as avg_transaction
FROM banking_catalog.silver.customers c
JOIN banking_catalog.silver.transactions t
ON c.customer_id = t.customer_id
GROUP BY c.customer_id, c.name
ORDER BY total_transaction_amount DESC
LIMIT 10;


customer_id,name,transaction_count,total_transaction_amount,avg_transaction
C1344,Customer344,1,49898.81,49898.81
C1018,Customer18,1,49792.88,49792.88
C1478,Customer478,1,49644.66,49644.66
C1097,Customer97,1,49571.36,49571.36
C1234,Customer234,1,49248.23,49248.23
C1476,Customer476,1,49225.12,49225.12
C1339,Customer339,1,48947.55,48947.55
C1260,Customer260,1,48757.81,48757.81
C1090,Customer90,1,48718.24,48718.24
C1125,Customer125,1,48610.39,48610.39



Branch Performance Comparison

In [0]:
%sql
-- Problem: How does each branch perform in terms of transactions?
SELECT 
    branch,
    COUNT(*) as transaction_count,
    SUM(amount) as total_amount,
    AVG(amount) as avg_amount,
    COUNT(DISTINCT customer_id) as unique_customers
FROM banking_catalog.silver.transactions
GROUP BY branch
ORDER BY total_amount DESC;


branch,transaction_count,total_amount,avg_amount,unique_customers
B4,25,803958.0799999998,32158.323199999995,25
B5,30,754118.7299999999,25137.290999999994,30
B15,37,740858.4299999999,20023.20081081081,37
B18,32,729879.0400000002,22808.720000000005,32
B7,27,717981.8600000001,26591.920740740745,27
B14,27,708950.95,26257.44259259259,27
B2,29,698653.8500000001,24091.51206896552,29
B11,26,687340.63,26436.178076923075,26
B13,28,655601.82,23414.350714285712,28
B3,23,622474.2899999999,27064.099565217388,23



Fraud Alert Status Distribution

In [0]:
%sql
-- Problem: What is the distribution of fraud alert statuses?
SELECT 
    status,
    COUNT(*) as alert_count,
    SUM(amount) as total_amount,
    AVG(amount) as avg_amount
FROM banking_catalog.silver.fraud_alerts
GROUP BY status
ORDER BY alert_count DESC;


status,alert_count,total_amount,avg_amount
INACTIVE,169,4065188.109999998,24054.367514792888
ACTIVE,167,4014259.6400000006,24037.482874251502
PENDING,164,4081816.6700000023,24889.12603658538



High-Risk Fraud Alerts

In [0]:
%sql
-- Problem: Identify high-risk fraud alerts (>40000)
SELECT 
    alert_id,
    customer_id,
    amount,
    status,
    type,
    date,
    branch
FROM banking_catalog.silver.fraud_alerts
WHERE amount > 40000
ORDER BY amount DESC;


alert_id,customer_id,amount,status,type,date,branch
AL394,C1393,49865.71,INACTIVE,LOAN,2023-08-08,B16
AL341,C1340,49583.45,PENDING,LOAN,2023-04-21,B12
AL74,C1073,49493.53,INACTIVE,CURRENT,2022-01-02,B9
AL436,C1435,49484.14,ACTIVE,LOAN,2022-03-30,B12
AL126,C1125,49335.07,ACTIVE,SAVINGS,2025-03-01,B7
AL62,C1061,49312.26,INACTIVE,CURRENT,2024-10-08,B18
AL260,C1259,49302.27,PENDING,SAVINGS,2021-10-02,B4
AL11,C1010,49188.7,ACTIVE,LOAN,2022-03-31,B13
AL200,C1199,49164.85,ACTIVE,SAVINGS,2024-07-10,B8
AL93,C1092,49089.1,PENDING,LOAN,2020-10-23,B13



Loan Portfolio Status

In [0]:
%sql
-- Problem: What is the status of the loan portfolio?
SELECT 
    status,
    COUNT(*) as loan_count,
    SUM(amount) as total_loan_amount,
    AVG(amount) as avg_loan_amount,
    ROUND(SUM(amount) * 100.0 / SUM(SUM(amount)) OVER(), 2) as percentage
FROM banking_catalog.silver.loans
GROUP BY status
ORDER BY total_loan_amount DESC;


status,loan_count,total_loan_amount,avg_loan_amount,percentage
PENDING,169,4161406.7000000016,24623.708284023676,34.28
ACTIVE,165,4027449.229999999,24408.783212121205,33.18
INACTIVE,166,3949589.4000000013,23792.70722891567,32.54


Customer Gender Distribution

In [0]:
%sql
-- Problem: What is the gender distribution of customers?
SELECT 
    COALESCE(gender, 'Unknown') as gender,
    COUNT(*) as customer_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
FROM banking_catalog.silver.customers
GROUP BY gender
ORDER BY percentage DESC;


gender,customer_count,percentage
M,169,33.80
F,166,33.20
U,165,33.00



Average Transaction Amount by City

In [0]:
%sql
-- Problem: Which cities have the highest average transaction amounts?
SELECT 
    c.city,
    COUNT(t.transaction_id) as transaction_count,
    ROUND(AVG(t.amount), 2) as avg_transaction_amount,
    ROUND(SUM(t.amount), 2) as total_transaction_amount
FROM banking_catalog.silver.customers c
JOIN banking_catalog.silver.transactions t
ON c.customer_id = t.customer_id
WHERE c.city IS NOT NULL
GROUP BY c.city
ORDER BY avg_transaction_amount DESC
LIMIT 10;


city,transaction_count,avg_transaction_amount,total_transaction_amount
Mumbai,123,26786.04,3294683.34
Pune,116,25747.6,2986721.26
Delhi,130,23979.54,3117339.77
Hyderabad,131,23404.04,3065928.74



Active vs Inactive Accounts

In [0]:
%sql
-- Problem: What is the ratio of active vs inactive accounts?
SELECT 
    status,
    COUNT(*) as account_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
FROM banking_catalog.silver.accounts
GROUP BY status
ORDER BY percentage DESC;


status,account_count,percentage
ACTIVE,173,34.60
INACTIVE,167,33.40
PENDING,160,32.00


Customer Lifecycle - Created Date Analysis

In [0]:
%sql
-- Problem: When were most customers created?
SELECT 
    YEAR(created_date) as year,
    MONTH(created_date) as month,
    COUNT(*) as new_customers
FROM banking_catalog.silver.customers
WHERE created_date IS NOT NULL
GROUP BY YEAR(created_date), MONTH(created_date)
ORDER BY year DESC, month DESC
LIMIT 12;


year,month,new_customers
2025,6,5
2025,5,8
2025,4,4
2025,3,12
2025,2,9
2025,1,11
2024,12,3
2024,11,5
2024,10,13
2024,9,5



Customers with Both Loans and Accounts

In [0]:
%sql
-- Problem: Which customers have both loans and active accounts?
SELECT DISTINCT
    c.customer_id,
    c.name,
    c.city
FROM banking_catalog.silver.customers c
WHERE EXISTS (
    SELECT 1 FROM banking_catalog.silver.accounts a 
    WHERE a.customer_id = c.customer_id AND a.status = 'ACTIVE'
)
AND EXISTS (
    SELECT 1 FROM banking_catalog.silver.loans l 
    WHERE l.customer_id = c.customer_id
);


customer_id,name,city
C1222,Customer222,Mumbai
C1303,Customer303,Mumbai
C1447,Customer447,Hyderabad
C1166,Customer166,Pune
C1285,Customer285,Hyderabad
C1117,Customer117,Pune
C1165,Customer165,Delhi
C1240,Customer240,Pune
C1498,Customer498,Delhi
C1208,Customer208,Mumbai



Fraudulent Transactions vs Regular Transactions

In [0]:
%sql
-- Problem: How do fraudulent transactions compare to regular ones?
SELECT 
    'Regular' as transaction_type,
    COUNT(*) as count,
    ROUND(AVG(amount), 2) as avg_amount,
    ROUND(SUM(amount), 2) as total_amount
FROM banking_catalog.silver.transactions
UNION ALL
SELECT 
    'Fraud' as transaction_type,
    COUNT(*) as count,
    ROUND(AVG(amount), 2) as avg_amount,
    ROUND(SUM(amount), 2) as total_amount
FROM banking_catalog.silver.fraud_alerts;


transaction_type,count,avg_amount,total_amount
Regular,500,24929.35,1.246467311E7
Fraud,500,24322.53,1.216126442E7



Quarterly Performance Analysis

In [0]:
%sql
-- Problem: How does the bank perform each quarter?
SELECT 
    YEAR(date) as year,
    QUARTER(date) as quarter,
    COUNT(*) as transaction_count,
    SUM(amount) as total_transactions,
    AVG(amount) as avg_transaction
FROM banking_catalog.silver.transactions
GROUP BY YEAR(date), QUARTER(date)
ORDER BY year DESC, quarter DESC;


year,quarter,transaction_count,total_transactions,avg_transaction
2025,2,20,589306.34,29465.317
2025,1,20,521312.0399999999,26065.601999999995
2024,4,22,606745.6299999998,27579.346818181806
2024,3,24,545186.1999999998,22716.09166666666
2024,2,14,324025.42000000004,23144.67285714286
2024,1,25,555739.64,22229.585600000002
2023,4,17,494640.68,29096.510588235295
2023,3,21,576699.43,27461.87761904762
2023,2,23,533691.7599999999,23203.989565217387
2023,1,20,444214.92,22210.746



Top 5 Branches by Customer Base


In [0]:
%sql
-- Problem: Which branches have the most customers?
SELECT 
    branch,
    COUNT(DISTINCT customer_id) as unique_customers,
    COUNT(*) as transaction_count
FROM banking_catalog.silver.transactions
GROUP BY branch
ORDER BY unique_customers DESC
LIMIT 5;


branch,unique_customers,transaction_count
B15,37,37
B18,32,32
B5,30,30
B17,29,29
B2,29,29



KYC Verification Status by City

In [0]:
%sql
-- Problem: How does KYC status vary across cities?
SELECT 
    city,
    COUNT(*) as total_customers,
    SUM(CASE WHEN kyc_status = 'Y' THEN 1 ELSE 0 END) as kyc_completed,
    ROUND(SUM(CASE WHEN kyc_status = 'Y' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as kyc_rate
FROM banking_catalog.silver.customers
WHERE city IS NOT NULL
GROUP BY city
ORDER BY kyc_rate DESC;


city,total_customers,kyc_completed,kyc_rate
Hyderabad,131,65,49.62
Pune,116,57,49.14
Delhi,130,57,43.85
Mumbai,123,50,40.65



Loan Default Risk Analysis

In [0]:
%sql
-- Problem: Identify customers with high loan default risk
SELECT 
    c.customer_id,
    c.name,
    COUNT(l.loan_id) as total_loans,
    SUM(l.amount) as total_loan_amount,
    ROUND(AVG(l.amount), 2) as avg_loan_amount,
    MAX(l.amount) as max_loan_amount
FROM banking_catalog.silver.customers c
JOIN banking_catalog.silver.loans l
ON c.customer_id = l.customer_id
WHERE l.status = 'PENDING'
GROUP BY c.customer_id, c.name
HAVING SUM(l.amount) > 100000
ORDER BY total_loan_amount DESC;


customer_id,name,total_loans,total_loan_amount,avg_loan_amount,max_loan_amount



Daily Transaction Pattern

In [0]:
%sql
-- Problem: What is the transaction pattern by day of week?
SELECT 
    DAYOFWEEK(date) as day_number,
    CASE DAYOFWEEK(date)
        WHEN 1 THEN 'Sunday'
        WHEN 2 THEN 'Monday'
        WHEN 3 THEN 'Tuesday'
        WHEN 4 THEN 'Wednesday'
        WHEN 5 THEN 'Thursday'
        WHEN 6 THEN 'Friday'
        WHEN 7 THEN 'Saturday'
    END as day_name,
    COUNT(*) as transaction_count,
    ROUND(AVG(amount), 2) as avg_amount
FROM banking_catalog.silver.transactions
GROUP BY DAYOFWEEK(date)
ORDER BY day_number;


day_number,day_name,transaction_count,avg_amount
1,Sunday,71,24691.08
2,Monday,58,22347.4
3,Tuesday,88,24483.26
4,Wednesday,66,22882.11
5,Thursday,68,27755.46
6,Friday,77,25799.96
7,Saturday,72,26065.88



Cross-Sell Analysis - Customers with Multiple Products

In [0]:
%sql
-- Problem: Which customers have the most banking products?
SELECT 
    c.customer_id,
    c.name,
    COUNT(DISTINCT a.account_id) as total_accounts,
    COUNT(DISTINCT l.loan_id) as total_loans,
    COUNT(DISTINCT f.alert_id) as fraud_alerts
FROM banking_catalog.silver.customers c
LEFT JOIN banking_catalog.silver.accounts a ON c.customer_id = a.customer_id
LEFT JOIN banking_catalog.silver.loans l ON c.customer_id = l.customer_id
LEFT JOIN banking_catalog.silver.fraud_alerts f ON c.customer_id = f.customer_id
GROUP BY c.customer_id, c.name
ORDER BY total_accounts DESC, total_loans DESC
LIMIT 10;


customer_id,name,total_accounts,total_loans,fraud_alerts
C1405,Customer405,1,1,1
C1093,Customer93,1,1,1
C1209,Customer209,1,1,1
C1320,Customer320,1,1,1
C1300,Customer300,1,1,1
C1486,Customer486,1,1,1
C1286,Customer286,1,1,1
C1440,Customer440,1,1,1
C1346,Customer346,1,1,1
C1409,Customer409,1,1,1



Branch Loan Performance error

In [0]:
%sql
-- Problem: How do branches perform on loan approvals?
SELECT 
    branch,
    COUNT(*) as total_loans,
    SUM(CASE WHEN status = 'ACTIVE' THEN 1 ELSE 0 END) as active_loans,
    SUM(amount) as total_loan_amount,
    ROUND(AVG(amount), 2) as avg_loan_amount,
    ROUND(SUM(CASE WHEN status = 'ACTIVE' THEN amount ELSE 0 END) * 100.0 / SUM(amount), 2) as active_loan_rate
FROM banking_catalog.silver.loans
GROUP BY branch
ORDER BY total_loan_amount DESC;


branch,total_loans,active_loans,total_loan_amount,avg_loan_amount,active_loan_rate
B2,35,10,854494.2500000003,24414.12,35.79
B4,33,11,831033.2399999999,25182.83,32.98
B7,33,12,821572.6900000001,24896.14,41.6
B12,29,13,733752.9899999999,25301.83,45.74
B14,28,10,701213.9800000001,25043.36,36.01
B1,25,11,701039.7300000001,28041.59,43.88
B11,27,6,665303.0,24640.85,28.41
B9,27,10,662837.0800000001,24549.52,37.15
B15,24,8,654161.0700000002,27256.71,30.93
B6,28,8,628523.4599999998,22447.27,29.0



Customer Wealth Segmentation

In [0]:
%sql
-- Problem: How is customer wealth distributed?
SELECT 
    CASE 
        WHEN total_balance < 10000 THEN 'Low (< 10K)'
        WHEN total_balance < 50000 THEN 'Medium (10K - 50K)'
        WHEN total_balance < 100000 THEN 'High (50K - 100K)'
        ELSE 'Very High (> 100K)'
    END as wealth_segment,
    COUNT(*) as customer_count,
    ROUND(AVG(total_balance), 2) as avg_balance,
    ROUND(SUM(total_balance), 2) as total_wealth
FROM (
    SELECT 
        customer_id,
        SUM(amount) as total_balance
    FROM banking_catalog.silver.accounts
    GROUP BY customer_id
) customer_balances
GROUP BY wealth_segment
ORDER BY wealth_segment;


wealth_segment,customer_count,avg_balance,total_wealth
Low (< 10K),117,4191.62,490419.0
Medium (10K - 50K),383,30525.96,1.169144264E7



Recent Fraud Trends

In [0]:
%sql
-- Problem: How is fraud trending over recent months?
SELECT 
    DATE_TRUNC('month', date) as month,
    COUNT(*) as fraud_count,
    ROUND(AVG(amount), 2) as avg_fraud_amount,
    SUM(amount) as total_fraud_amount
FROM banking_catalog.silver.fraud_alerts
WHERE date >= DATE_SUB(CURRENT_DATE(), 180)
GROUP BY DATE_TRUNC('month', date)
ORDER BY month DESC;


month,fraud_count,avg_fraud_amount,total_fraud_amount



Customer Acquisition by State

In [0]:
%sql
-- Problem: Which states have the highest customer acquisition?
SELECT 
    state,
    COUNT(*) as total_customers,
    SUM(CASE WHEN kyc_status = 'Y' THEN 1 ELSE 0 END) as kyc_verified,
    DATE_TRUNC('year', created_date) as acquisition_year
FROM banking_catalog.silver.customers
WHERE state IS NOT NULL
GROUP BY state, DATE_TRUNC('year', created_date)
ORDER BY acquisition_year DESC, total_customers DESC;


state,total_customers,kyc_verified,acquisition_year
STATE,49,19,2025-01-01T00:00:00.000Z
STATE,82,38,2024-01-01T00:00:00.000Z
STATE,81,38,2023-01-01T00:00:00.000Z
STATE,102,47,2022-01-01T00:00:00.000Z
STATE,82,40,2021-01-01T00:00:00.000Z
STATE,104,47,2020-01-01T00:00:00.000Z



Transaction Amount Distribution

In [0]:
%sql
-- Problem: How are transaction amounts distributed?
SELECT 
    CASE 
        WHEN amount < 1000 THEN 'Micro (< 1K)'
        WHEN amount < 10000 THEN 'Small (1K - 10K)'
        WHEN amount < 50000 THEN 'Medium (10K - 50K)'
        ELSE 'Large (> 50K)'
    END as transaction_segment,
    COUNT(*) as transaction_count,
    ROUND(AVG(amount), 2) as avg_amount,
    ROUND(SUM(amount), 2) as total_amount,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
FROM banking_catalog.silver.transactions
GROUP BY transaction_segment
ORDER BY transaction_segment;


transaction_segment,transaction_count,avg_amount,total_amount,percentage
Medium (10K - 50K),391,30672.37,1.199289651E7,78.20
Micro (< 1K),22,204.35,4495.62,4.40
Small (1K - 10K),87,5371.05,467280.98,17.40



Customer Activity Patterns

In [0]:
%sql
-- Problem: What are the activity patterns of customers?
WITH customer_stats AS (
    SELECT 
        customer_id,
        COUNT(*) as total_transactions,
        SUM(amount) as total_spent,
        DATEDIFF(CURRENT_DATE(), MAX(date)) as days_since_last_transaction
    FROM banking_catalog.silver.transactions
    GROUP BY customer_id
)
SELECT 
    CASE 
        WHEN total_transactions >= 100 THEN 'Very Active'
        WHEN total_transactions >= 50 THEN 'Active'
        WHEN total_transactions >= 10 THEN 'Moderate'
        ELSE 'Occasional'
    END as activity_level,
    COUNT(*) as customer_count,
    ROUND(AVG(total_spent), 2) as avg_spent,
    ROUND(AVG(days_since_last_transaction), 2) as avg_days_inactive
FROM customer_stats
GROUP BY activity_level
ORDER BY activity_level;


activity_level,customer_count,avg_spent,avg_days_inactive
Occasional,500,24929.35,1472.12



YOY -Year-over-Year Analysis - Customer Growth

In [0]:
%sql
-- Problem: How has customer base grown year over year?
SELECT 
    YEAR(created_date) as year,
    COUNT(*) as new_customers,
    LAG(COUNT(*)) OVER (ORDER BY YEAR(created_date)) as prev_year_customers,
    ROUND((COUNT(*) - LAG(COUNT(*)) OVER (ORDER BY YEAR(created_date))) * 100.0 / LAG(COUNT(*)) OVER (ORDER BY YEAR(created_date)), 2) as yoy_growth
FROM banking_catalog.silver.customers
WHERE created_date IS NOT NULL
GROUP BY YEAR(created_date)
ORDER BY year;


year,new_customers,prev_year_customers,yoy_growth
2020,104,null,null
2021,82,104,-21.15
2022,102,82,24.39
2023,81,102,-20.59
2024,82,81,1.23
2025,49,82,-40.24



## OPTIMIZATION & MAINTENANCE



Optimize Delta Tables

In [0]:
%sql
-- Optimize all tables for better performance
OPTIMIZE banking_catalog.silver.accounts;
OPTIMIZE banking_catalog.silver.transactions;
OPTIMIZE banking_catalog.silver.loans;
OPTIMIZE banking_catalog.gold.fact_transactions;
OPTIMIZE banking_catalog.gold.fact_loans;


path,metrics
abfss://gold@stbankinglake005.dfs.core.windows.net/fact_loans,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1786789730075, 1786789730953, 8, 0, null, List(0, 0), null, 10, 10, 0, 0, null, null)"



Vacuum Old Files

In [0]:
%sql
-- Clean up old files (retention period 7 days)
VACUUM banking_catalog.silver.transactions RETAIN 168 HOURS;
VACUUM banking_catalog.gold.fact_transactions RETAIN 168 HOURS;
VACUUM banking_catalog.bronze.transactions RETAIN 168 HOURS;


path
abfss://bronze@stbankinglake005.dfs.core.windows.net/transactions



Collect Table Statistics

In [0]:
%sql
-- Collect statistics for query optimization
ANALYZE TABLE banking_catalog.silver.accounts COMPUTE STATISTICS;
ANALYZE TABLE banking_catalog.silver.transactions COMPUTE STATISTICS;
ANALYZE TABLE banking_catalog.gold.fact_transactions COMPUTE STATISTICS;
